# MMTFv3 Detailed Backtest

Loads the best model and hyperparameters saved by **`mmtfv3_cl_gc_optuna.ipynb`**
and produces a full trading backtest covering:

| Section | Metrics |
|---|---|
| Portfolio summary | Sharpe, Sortino, Calmar (annualised), profit factor, max-DD |
| Position evolution | Daily avg weight across time, rolling exposure, per-ticker heatmap |
| Calibration | Position vs realized-return quantile, position–return scatter |
| Trade-level | Avg profit / trade, holding duration, long vs short breakdown |
| Rolling | 30-day rolling Sharpe, rolling drawdown |
| Calendar | Monthly PnL heatmap |
| TC-adjusted | All key metrics repeated net of transaction costs |

Run the training notebook first to generate the checkpoint and `best_params.json`.

## 1. Config

In [ ]:
from pathlib import Path
import numpy as np

# ── Must match training notebook ────────────────────────────────────────────
TICKERS                 = ['CL', 'GC']
TUNE_BACKBONE           = True          # how prefix was built
BACKBONE                = 'mamba'       # fallback if TUNE_BACKBONE=False
USE_PTP                 = True
BAR_MINUTES             = 5
TARGET_HORIZON_MINUTES  = 30
SAMPLE_SESSION          = "usa"
SAMPLE_SESSION_START    = None
SAMPLE_SESSION_END      = None
SAMPLE_STRIDE           = 6
AE_WINDOW               = 21

IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT    = Path('/content/drive/MyDrive/features')
    RESULTS_PATH = Path('/content/drive/MyDrive/results/mmtfv3_cl_gc')
else:
    DATA_ROOT    = Path('/workspace/model_data')
    RESULTS_PATH = Path('/workspace/results/mmtfv3_cl_gc')

BACKTEST_PATH = RESULTS_PATH          # where backtest artefacts are written
BACKTEST_PATH.mkdir(parents=True, exist_ok=True)

# ── Backtest params ──────────────────────────────────────────────────────────
BACKTEST_BATCH_SIZE = 256
TRADE_THRESH        = 0.05            # |position| > threshold = "active bar"
TC_COST_BPS         = 0.5            # one-way transaction cost in basis points
TC_COST             = TC_COST_BPS * 1e-4

# Session: 08:30-16:00 → 450 min → 15 bars of 30 min each
BARS_PER_DAY   = int(450 / TARGET_HORIZON_MINUTES)   # 15
BARS_PER_YEAR  = 252 * BARS_PER_DAY                  # 3780
ANNUALIZATION  = np.sqrt(BARS_PER_YEAR)               # ≈61.5

# ── Prefix (must match training notebook) ───────────────────────────────────
bb_tag  = "tuned" if TUNE_BACKBONE else BACKBONE
ptp_tag = "_ptp" if USE_PTP else ""
prefix  = f"{'_'.join(TICKERS)}_mmtfv3_{bb_tag}{ptp_tag}_optuna"
print(f"prefix          : {prefix}")
print(f"RESULTS_PATH    : {RESULTS_PATH}")
print(f"BARS_PER_DAY    : {BARS_PER_DAY}   ANNUALIZATION: {ANNUALIZATION:.2f}")

## 2. Imports

In [ ]:
import json, warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
from torch.utils.data import DataLoader
warnings.filterwarnings("ignore")

from CTAFlow.data.datasets.v3_continuous import (
    V3ContinuousPrep,
    V3ContinuousDataset,
    build_v3_loaders,
    unpack_v3_batch,
    v3_collate_fn,
    SessionSpec,
)
from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import (
    MMTFv3Core,
    StatefulMMTFv3Core,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 3. Load Saved Artefacts

In [ ]:
# ── Best hyperparameters ─────────────────────────────────────────────────────
params_path = RESULTS_PATH / f"{prefix}_best_params.json"
with open(params_path) as f:
    best = json.load(f)

print("Best hyperparameters loaded:")
for k, v in sorted(best.items()):
    print(f"  {k:35s}: {v}")

# ── (Optional) Optuna study for trial history ────────────────────────────────
import joblib
study_path = RESULTS_PATH / f"{prefix}_study.pkl"
study = None
if study_path.exists():
    study = joblib.load(study_path)
    print(f"\nOptuna study loaded  — {len(study.trials)} trials")
    print(f"  Best value : {study.best_value:.6f}")
    print(f"  Best trial : #{study.best_trial.number}")

## 4. Reconstruct Data Prep & Model

In [ ]:
# ── V3ContinuousPrep ─────────────────────────────────────────────────────────
prep = V3ContinuousPrep.from_directories(
    root_dir=DATA_ROOT,
    tickers=TICKERS,
    sessions=[SessionSpec("USA", "08:30", "16:00")],
    bar_minutes=BAR_MINUTES,
    target_horizon_minutes=TARGET_HORIZON_MINUTES,
    ae_window=AE_WINDOW,
)
dims = prep.get_dims()
F_TECH          = dims['f_tech']
F_SEQ           = dims['f_seq']
F_AE            = 4
NUMBARS_CHANNELS = dims['numbars_channels']
VPIN_TIME        = dims['vpin_time']
VPIN_CHANNELS    = dims['vpin_channels']
VPIN_BINS        = dims['vpin_bins']
print("Dims:", dims)

# ── Reconstruct MMTFv3Core ────────────────────────────────────────────────────
backbone = best.get('backbone', BACKBONE)
base_model = MMTFv3Core(
    f_tech=F_TECH, f_seq=F_SEQ, f_ae=F_AE,
    ae_type=best.get('ae_type', 'vae'),
    d_latent=best['d_latent'], d_ae_hidden=best['d_ae_hidden'],
    kl_weight=best['kl_weight'], recon_weight=best['recon_weight'],
    n_tickers=prep.n_tickers,
    n_asset_classes=prep.n_asset_classes,
    n_asset_subclasses=prep.n_asset_subclasses,
    d_model=best['d_model'], d_static_emb=best['d_static_emb'],
    backbone=backbone, n_heads=best['n_heads'], n_layers=best['n_layers'],
    d_ff=best.get('d_ff', 512), d_state=best.get('d_state', 16),
    d_conv=best.get('d_conv', 4), expand=best.get('expand', 2),
    dropout=best['dropout'], grn_dropout=best['grn_dropout'],
    numbars_channels=NUMBARS_CHANNELS, vpin_channels=VPIN_CHANNELS,
    vpin_bins=VPIN_BINS, vpin_time=VPIN_TIME,
)

model = StatefulMMTFv3Core(
    base_model=base_model,
    n_tickers=prep.n_tickers,
    quantile_head=USE_PTP,
    ptp_temperature=best.get('ptp_temperature', 1.5),
    state_hidden_dim=best.get('state_hidden_dim', 16),
    state_momentum=best.get('state_momentum', 0.9),
    update_on_eval=True,
).to(device)

# ── Load checkpoint ───────────────────────────────────────────────────────────
ckpt_path = RESULTS_PATH / f"{prefix}_best_model.pth"
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()
print(f"\nCheckpoint loaded from {ckpt_path}")
print(f"  Trained metrics: {ckpt.get('metrics', {})}")
n_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters     : {n_params:,}")

## 5. Build Date-Aware Dataset

`v3_collate_fn` drops the `date` field, so we extract dates from the raw sample
list **before** wrapping in a DataLoader. Because `shuffle=False`, sample order
is preserved and we can align dates by running index.

In [ ]:
tech_lookback = int(best['tech_lookback'])
seq_lookback  = int(best['seq_lookback'])

# Build all samples (includes 'date' and 'ticker' fields)
all_samples = prep.build_samples(
    tech_lookback=tech_lookback,
    seq_lookback_bars=seq_lookback,
    stride=SAMPLE_STRIDE,
    sample_session=SAMPLE_SESSION,
    sample_session_start=SAMPLE_SESSION_START,
    sample_session_end=SAMPLE_SESSION_END,
)
print(f"Total samples: {len(all_samples):,}")

# ── Extract metadata before collation drops it ───────────────────────────────
sample_dates   = [s['date']   for s in all_samples]   # datetime.date
sample_tickers = [s['ticker'] for s in all_samples]   # str, e.g. 'CL'

# ── Val / in-sample split (mirror of training notebook) ──────────────────────
VAL_RATIO   = 0.2
split_idx   = int(len(all_samples) * (1 - VAL_RATIO))
is_val      = np.array([i >= split_idx for i in range(len(all_samples))])
print(f"In-sample  : {split_idx:,} samples  ({sample_dates[0]} → {sample_dates[split_idx-1]})")
print(f"Out-of-sample: {len(all_samples)-split_idx:,} samples ({sample_dates[split_idx]} → {sample_dates[-1]})")

# ── DataLoader (no shuffle) ───────────────────────────────────────────────────
dataset     = V3ContinuousDataset(all_samples)
full_loader = DataLoader(
    dataset, batch_size=BACKTEST_BATCH_SIZE,
    shuffle=False, collate_fn=v3_collate_fn, num_workers=0,
)
print(f"DataLoader  : {len(full_loader)} batches of up to {BACKTEST_BATCH_SIZE}")

## 6. Inference Pass

In [ ]:
bt_pos_list, bt_ret_list, bt_tid_list, bt_logits_list = [], [], [], []
running_idx = 0

model.eval()
if hasattr(model, 'reset_position_state'):
    model.reset_position_state()

with torch.no_grad():
    for batch in full_loader:
        inputs, targets = unpack_v3_batch(batch, device=device)
        out = model(**inputs, return_ae_losses=True)

        if USE_PTP:
            position, _ae_losses, logits = out   # (B,1), dict, (B,5)
            bt_logits_list.append(logits.cpu())
        else:
            position, _ae_losses = out

        bt_pos_list.append(position.view(-1).cpu())
        bt_ret_list.append(targets.view(-1).float().cpu())
        bt_tid_list.append(inputs['ticker_id'].view(-1).cpu())
        running_idx += targets.shape[0]

bt_pos    = torch.cat(bt_pos_list).numpy()    # (N,)
bt_ret    = torch.cat(bt_ret_list).numpy()    # (N,)
bt_tid    = torch.cat(bt_tid_list).numpy()    # (N,)
bt_logits = torch.cat(bt_logits_list).numpy() if bt_logits_list else None  # (N,5)

id_to_ticker = {meta.ticker_id: t for t, meta in prep.registry.items()}
print(f"Inference done: {len(bt_pos):,} samples")
print(f"Position range: [{bt_pos.min():.4f}, {bt_pos.max():.4f}]  mean={bt_pos.mean():.4f}")

## 7. Backtest DataFrame

In [ ]:
import pandas as pd

bt = pd.DataFrame({
    'date'        : sample_dates,
    'ticker'      : sample_tickers,
    'ticker_id'   : bt_tid.astype(int),
    'position'    : bt_pos,
    'fwd_return'  : bt_ret,
    'is_val'      : is_val,
})

bt['date']         = pd.to_datetime(bt['date'])
bt['strategy_ret'] = bt['position'] * bt['fwd_return']
bt['active']       = bt['position'].abs() > TRADE_THRESH

# ── Transaction cost: charge TC_COST on every direction flip ─────────────────
bt = bt.sort_values(['ticker', 'date']).reset_index(drop=True)
signs           = np.sign(bt['position'].values)
ticker_boundary = (bt['ticker'] != bt['ticker'].shift(1)).values
sign_flip       = np.concatenate([[False], np.diff(signs) != 0])
sign_flip[ticker_boundary] = False          # don't count cross-ticker as flip

bt['tc_cost']         = np.where(sign_flip, TC_COST, 0.0)
bt['strategy_ret_tc'] = bt['strategy_ret'] - bt['tc_cost']
bt['is_trade']        = sign_flip

# ── Cumulative PnL (gross & TC-adjusted) per ticker ──────────────────────────
for col in ('strategy_ret', 'strategy_ret_tc'):
    bt[f'cum_{col}'] = bt.groupby('ticker')[col].cumsum()

print(f"Columns: {list(bt.columns)}")
print(bt.head(3).to_string())

## 8. Portfolio-Level Metrics

In [ ]:
def _metrics(df, label, col='strategy_ret'):
    """Compute trading metrics for a slice of the backtest DataFrame."""
    sr     = df[col].values
    pos    = df['position'].values
    ret    = df['fwd_return'].values
    active = df['active'].values

    cum     = np.cumsum(sr)
    mean_sr = sr.mean()
    std_sr  = sr.std() + 1e-8
    neg     = sr[sr < 0]
    dside   = float(np.sqrt((neg**2).mean())) if len(neg) > 0 else 1e-8
    gp      = sr[sr > 0].sum()
    gl      = np.abs(sr[sr < 0]).sum() + 1e-9
    run_max = np.maximum.accumulate(cum)
    mdd     = float((run_max - cum).max())

    correct = ((pos > 0) & (ret > 0)) | ((pos < 0) & (ret < 0))
    dir_acc = (correct & active).sum() / max(active.sum(), 1)
    win_rate = ((sr[active] > 0).mean() * 100) if active.sum() > 0 else 0.0

    sharpe_ann  = (mean_sr / std_sr) * ANNUALIZATION
    sortino_ann = (mean_sr / (dside + 1e-8)) * ANNUALIZATION
    annual_ret  = mean_sr * BARS_PER_YEAR
    calmar      = annual_ret / (mdd + 1e-9)

    n_trades = int(df['is_trade'].sum())
    avg_trade = df.loc[df['is_trade'], col].mean() if n_trades > 0 else 0.0

    return {
        'Label'           : label,
        'N Samples'       : len(sr),
        'Net PnL'         : round(float(sr.sum()), 5),
        'Ann. Return'     : round(float(annual_ret), 5),
        'Sharpe (ann)'    : round(float(sharpe_ann), 3),
        'Sortino (ann)'   : round(float(sortino_ann), 3),
        'Calmar'          : round(float(calmar), 3),
        'Win Rate (%)'    : round(float(win_rate), 2),
        'Dir Acc (%)'     : round(float(dir_acc * 100), 2),
        'Profit Factor'   : round(float(gp / gl), 4),
        'Max Drawdown'    : round(mdd, 5),
        '# Trades'        : n_trades,
        'Avg Trade PnL'   : round(float(avg_trade), 6),
        'Avg |Position|'  : round(float(np.abs(pos).mean()), 4),
        'Active Bars (%)'  : round(float(active.mean() * 100), 2),
    }

# ── Build table ───────────────────────────────────────────────────────────────
rows = []
for split_label, split_mask in [('IS (train)', ~bt['is_val']), ('OOS (val)', bt['is_val'])]:
    df_split = bt[split_mask]
    rows.append(_metrics(df_split, f'ALL — {split_label}'))
    for tid in sorted(id_to_ticker):
        tk = id_to_ticker[tid]
        df_tk = df_split[df_split['ticker'] == tk]
        if len(df_tk) == 0: continue
        rows.append(_metrics(df_tk, f'{tk} — {split_label}'))
    rows.append({k: '─' * 6 if k != 'Label' else '──────────────────────' for k in rows[0]})

# TC-adjusted summary (OOS only)
oos = bt[bt['is_val']]
rows.append({k: '' if k != 'Label' else '=== OOS  TC-adjusted ===' for k in rows[0]})
rows.append(_metrics(oos, 'ALL — OOS (TC adj)', col='strategy_ret_tc'))
for tid in sorted(id_to_ticker):
    tk = id_to_ticker[tid]
    df_tk = oos[oos['ticker'] == tk]
    if len(df_tk) == 0: continue
    rows.append(_metrics(df_tk, f'{tk} — OOS (TC adj)', col='strategy_ret_tc'))

df_summary = pd.DataFrame(rows).set_index('Label')
print("=" * 100)
print("PORTFOLIO BACKTEST SUMMARY — MMTFv3 Best Model")
print("=" * 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print(df_summary.to_string())

## 9. Position (Weight) Evolution Over Time

Daily average position per ticker and the full portfolio. Rolling 5-day mean
shows the medium-term directional bias.

In [ ]:
# ── Daily average position per ticker ────────────────────────────────────────
daily_pos = (
    bt.groupby(['date', 'ticker'])['position']
    .mean()
    .unstack('ticker')
    .fillna(0)
)
daily_pos['COMBINED'] = daily_pos.mean(axis=1)  # equal-weight portfolio

# ── Rolling 5-day average ─────────────────────────────────────────────────────
roll5 = daily_pos.rolling(5, min_periods=1).mean()

palette = {'CL': '#2980b9', 'GC': '#e67e22', 'COMBINED': 'black'}

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# 1. Raw daily position per ticker
ax = axes[0]
for col in [c for c in daily_pos.columns if c != 'COMBINED']:
    ax.plot(daily_pos.index, daily_pos[col], alpha=0.5, lw=0.8,
            color=palette.get(col), label=col)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Daily Average Position by Ticker')
ax.set_ylabel('Avg Position')
ax.legend()
ax.grid(True, alpha=0.25)

# 2. Rolling 5-day + OOS shading
ax = axes[1]
for col in daily_pos.columns:
    ax.plot(roll5.index, roll5[col], lw=1.5,
            color=palette.get(col, 'gray'), label=col)
# OOS shading
oos_start = bt.loc[bt['is_val'], 'date'].min()
ax.axvspan(oos_start, daily_pos.index[-1], alpha=0.07, color='green', label='OOS')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Rolling 5-Day Avg Position (OOS = green shaded)')
ax.set_ylabel('Avg Position')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# 3. Exposure heatmap (ticker × month)
ax = axes[2]
hm_data = (
    bt.assign(ym=bt['date'].dt.to_period('M'))
    .groupby(['ym', 'ticker'])['position']
    .apply(lambda x: x.abs().mean())
    .unstack('ticker')
)
hm_data.index = hm_data.index.astype(str)
im = ax.imshow(hm_data.T.values, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=hm_data.values.max())
ax.set_xticks(range(len(hm_data.index)))
ax.set_xticklabels(hm_data.index, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(len(hm_data.columns)))
ax.set_yticklabels(hm_data.columns)
ax.set_title('Monthly Avg Absolute Position (Exposure) Heatmap')
plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01)

plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_position_evolution.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Calibration — Position vs Realized Return Quantile

Bin realized forward returns into deciles and measure the model's average
**position** and **strategy return** in each bin. A well-calibrated model
assigns large positive (negative) positions to the top (bottom) return deciles.

In [ ]:
N_QUANTILES = 10

fig, axes = plt.subplots(2, len(TICKERS) + 1, figsize=(7 * (len(TICKERS) + 1), 10))

for col_idx, (label, df_slice) in enumerate(
    [('ALL', bt)] + [(tk, bt[bt['ticker'] == tk]) for tk in sorted(id_to_ticker.values())]
):
    df_q = df_slice.copy()
    df_q['ret_q'] = pd.qcut(df_q['fwd_return'], q=N_QUANTILES, labels=False)

    agg = df_q.groupby('ret_q').agg(
        avg_position=('position', 'mean'),
        avg_strat_ret=('strategy_ret', 'mean'),
        avg_fwd_ret=('fwd_return', 'mean'),
        count=('position', 'count'),
    ).reset_index()

    q_labels = [f"D{i+1}" for i in range(N_QUANTILES)]
    cmap_vals = plt.cm.RdYlGn(np.linspace(0, 1, N_QUANTILES))

    # Top: avg position per decile
    ax = axes[0, col_idx]
    bars = ax.bar(q_labels, agg['avg_position'].values, color=cmap_vals, alpha=0.9)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{label} — Avg Position by Return Decile')
    ax.set_xlabel('Return Decile (D1=worst, D10=best)')
    ax.set_ylabel('Avg Position')
    ax.grid(True, alpha=0.25, axis='y')

    # Bottom: avg strategy return per decile
    ax = axes[1, col_idx]
    ax.bar(q_labels, agg['avg_strat_ret'].values, color=cmap_vals, alpha=0.9)
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{label} — Avg Strategy Return by Decile')
    ax.set_xlabel('Return Decile')
    ax.set_ylabel('Avg Strategy Return')
    ax.grid(True, alpha=0.25, axis='y')

plt.suptitle('Position Calibration Across Realized Return Quantiles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_calibration.png", dpi=150, bbox_inches='tight')
plt.show()

## 11. Position–Return Scatter

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(8 * len(TICKERS), 5), sharey=False)

for ax, tk in zip(axes if len(TICKERS) > 1 else [axes], sorted(id_to_ticker.values())):
    df_tk = bt[bt['ticker'] == tk].sample(n=min(5000, len(bt[bt['ticker'] == tk])),
                                           random_state=42)
    sc = ax.scatter(
        df_tk['position'], df_tk['fwd_return'],
        c=df_tk['strategy_ret'], cmap='RdYlGn',
        alpha=0.35, s=8, vmin=-0.003, vmax=0.003,
    )
    # Regression line
    from numpy.polynomial import polynomial as P
    coefs = np.polyfit(df_tk['position'], df_tk['fwd_return'], 1)
    xs = np.linspace(df_tk['position'].min(), df_tk['position'].max(), 100)
    ax.plot(xs, np.polyval(coefs, xs), 'k--', lw=1.5, label=f"slope={coefs[0]:.4f}")
    ax.axhline(0, color='gray', lw=0.8, linestyle=':')
    ax.axvline(0, color='gray', lw=0.8, linestyle=':')
    ax.set_title(f'{tk} — Position vs Realized Return')
    ax.set_xlabel('Position')
    ax.set_ylabel('Realized Forward Return')
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='Strategy Ret')

plt.suptitle('Position–Return Scatter (colour = strategy return)', fontsize=13)
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. Trade-Level Analysis

A "trade" is defined as a contiguous run of bars sharing the same
non-flat sign. We compute:
- Avg profit per trade
- Avg holding duration (bars)
- Long vs short breakdown

In [ ]:
def extract_trades(df_ticker):
    """Extract trade records from a single-ticker DataFrame ordered by time."""
    df = df_ticker.sort_values('date').reset_index(drop=True)
    pos = df['position'].values
    ret = df['strategy_ret_tc'].values
    active = np.abs(pos) > TRADE_THRESH

    trades = []
    in_trade = False
    t_start = None
    t_sign  = 0
    t_ret   = []

    for i in range(len(pos)):
        sign_i = int(np.sign(pos[i]))
        if active[i]:
            if not in_trade or sign_i != t_sign:
                # close previous trade if open
                if in_trade and t_ret:
                    trades.append({'sign': t_sign, 'n_bars': len(t_ret),
                                   'gross_ret': sum(t_ret)})
                # open new trade
                in_trade = True
                t_sign   = sign_i
                t_ret    = [ret[i]]
            else:
                t_ret.append(ret[i])
        else:
            if in_trade and t_ret:
                trades.append({'sign': t_sign, 'n_bars': len(t_ret),
                               'gross_ret': sum(t_ret)})
            in_trade = False
            t_ret    = []
            t_sign   = 0

    if in_trade and t_ret:
        trades.append({'sign': t_sign, 'n_bars': len(t_ret), 'gross_ret': sum(t_ret)})

    return pd.DataFrame(trades) if trades else pd.DataFrame(
        columns=['sign', 'n_bars', 'gross_ret'])

# ── Collect trades per ticker ─────────────────────────────────────────────────
trade_rows = []
fig, axes = plt.subplots(2, len(TICKERS), figsize=(8 * len(TICKERS), 10))

for col_idx, tk in enumerate(sorted(id_to_ticker.values())):
    for split_label, split_mask in [('IS', ~bt['is_val']), ('OOS', bt['is_val'])]:
        df_tk = bt[(bt['ticker'] == tk) & split_mask]
        trades = extract_trades(df_tk)
        if len(trades) == 0: continue

        wins  = trades['gross_ret'] > 0
        longs = trades['sign']  == 1
        row = {
            'Ticker'           : tk,
            'Split'            : split_label,
            '# Trades'         : len(trades),
            '# Long'           : int(longs.sum()),
            '# Short'          : int((~longs).sum()),
            'Avg Profit/Trade' : round(trades['gross_ret'].mean(), 6),
            'Long Avg Profit'  : round(trades.loc[longs,  'gross_ret'].mean(), 6) if longs.any() else 0,
            'Short Avg Profit' : round(trades.loc[~longs, 'gross_ret'].mean(), 6) if (~longs).any() else 0,
            'Win Rate (%)'     : round(wins.mean() * 100, 2),
            'Long Win (%)'     : round(wins[longs].mean()  * 100, 2) if longs.any()  else 0,
            'Short Win (%)'    : round(wins[~longs].mean() * 100, 2) if (~longs).any() else 0,
            'Avg Hold (bars)'  : round(trades['n_bars'].mean(), 1),
            'Best Trade'       : round(trades['gross_ret'].max(), 6),
            'Worst Trade'      : round(trades['gross_ret'].min(), 6),
        }
        trade_rows.append(row)

    # ── Visualise OOS trades ──────────────────────────────────────────────────
    oos_trades = extract_trades(bt[(bt['ticker'] == tk) & bt['is_val']])

    ax0 = axes[0, col_idx] if len(TICKERS) > 1 else axes[0]
    ax1 = axes[1, col_idx] if len(TICKERS) > 1 else axes[1]

    if len(oos_trades) > 0:
        colors = ['#27ae60' if v > 0 else '#e74c3c' for v in oos_trades['gross_ret']]
        ax0.bar(range(len(oos_trades)), oos_trades['gross_ret'].values,
                color=colors, alpha=0.75, width=1.0)
        ax0.axhline(0, color='gray', lw=0.8, linestyle=':')
        ax0.set_title(f'{tk} OOS — Trade PnL (TC-adj)')
        ax0.set_xlabel('Trade #')
        ax0.set_ylabel('Gross Return')
        ax0.grid(True, alpha=0.25, axis='y')

        ax1.hist(oos_trades['n_bars'].values, bins=30,
                 color='steelblue', alpha=0.8, edgecolor='white')
        ax1.axvline(oos_trades['n_bars'].mean(), color='red', lw=1.5,
                    linestyle='--', label=f"mean={oos_trades['n_bars'].mean():.1f}")
        ax1.set_title(f'{tk} OOS — Holding Duration (bars)')
        ax1.set_xlabel('Bars Held')
        ax1.set_ylabel('Count')
        ax1.legend()
        ax1.grid(True, alpha=0.25)

df_trades = pd.DataFrame(trade_rows).set_index(['Ticker', 'Split'])
print("\nTRADE-LEVEL ANALYSIS")
print("=" * 90)
print(df_trades.to_string())

plt.suptitle('Trade-Level Analysis (OOS, TC-adjusted)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_trades.png", dpi=150, bbox_inches='tight')
plt.show()

## 13. Cumulative PnL & Rolling Sharpe

In [ ]:
ROLL_WINDOW = max(BARS_PER_DAY * 21, 200)   # ~21 trading days

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=False)

# ── 1. Cumulative gross PnL (time-indexed, per ticker + combined) ─────────────
ax = axes[0]
palette_tk = plt.cm.tab10(np.linspace(0, 0.4, len(TICKERS)))
for tid, color in zip(sorted(id_to_ticker), palette_tk):
    tk   = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum   = df_tk['strategy_ret'].cumsum().values
    ax.plot(df_tk['date'].values, cum, lw=1.5, color=color, alpha=0.85, label=tk)

# Combined (sum across tickers on same bars)
comb_daily = (
    bt.groupby('date')['strategy_ret'].sum()
    .sort_index().cumsum()
)
ax.plot(comb_daily.index, comb_daily.values, lw=2.2, color='black',
        linestyle='--', label='Combined')
ax.axvline(bt.loc[bt['is_val'], 'date'].min(), color='green',
           lw=1.5, linestyle=':', label='OOS start')
ax.fill_between(comb_daily.index, comb_daily.values, 0,
                where=comb_daily.values >= 0, color='green', alpha=0.06)
ax.fill_between(comb_daily.index, comb_daily.values, 0,
                where=comb_daily.values < 0, color='red', alpha=0.06)
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title('Cumulative Gross PnL')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# ── 2. TC-adjusted cumulative PnL ────────────────────────────────────────────
ax = axes[1]
for tid, color in zip(sorted(id_to_ticker), palette_tk):
    tk   = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum   = df_tk['strategy_ret_tc'].cumsum().values
    ax.plot(df_tk['date'].values, cum, lw=1.5, color=color, alpha=0.85, label=f'{tk} (TC)')

comb_tc = bt.groupby('date')['strategy_ret_tc'].sum().sort_index().cumsum()
ax.plot(comb_tc.index, comb_tc.values, lw=2.2, color='black', linestyle='--',
        label='Combined (TC)')
ax.axvline(bt.loc[bt['is_val'], 'date'].min(), color='green', lw=1.5, linestyle=':')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.set_title(f'Cumulative TC-Adjusted PnL  ({TC_COST_BPS} bps per leg)')
ax.set_ylabel('Cumulative PnL')
ax.legend(ncol=4)
ax.grid(True, alpha=0.25)

# ── 3. Rolling Sharpe (combined, bar-level) ────────────────────────────────────
ax = axes[2]
comb_sr  = bt.sort_values('date').groupby('date')['strategy_ret'].sum()
roll_mu  = comb_sr.rolling(ROLL_WINDOW, min_periods=BARS_PER_DAY * 5).mean()
roll_std = comb_sr.rolling(ROLL_WINDOW, min_periods=BARS_PER_DAY * 5).std() + 1e-8
roll_sh  = (roll_mu / roll_std) * ANNUALIZATION

ax.plot(roll_sh.index, roll_sh.values, color='steelblue', lw=1.2, label='Rolling Sharpe')
ax.axhline(0, color='gray', lw=0.8, linestyle=':')
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values >= 0, color='steelblue', alpha=0.15)
ax.fill_between(roll_sh.index, roll_sh.values, 0,
                where=roll_sh.values < 0, color='red', alpha=0.15)
ax.axvline(bt.loc[bt['is_val'], 'date'].min(), color='green', lw=1.5,
           linestyle=':', label='OOS start')
ax.set_title(f'Rolling Annualised Sharpe (window={ROLL_WINDOW} bars ≈ 21 days)')
ax.set_ylabel('Sharpe')
ax.legend()
ax.grid(True, alpha=0.25)

plt.suptitle('MMTFv3 Backtest — Cumulative PnL & Rolling Sharpe',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_cum_pnl.png", dpi=150, bbox_inches='tight')
plt.show()

## 14. Drawdown Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=False)

# ── Combined drawdown curve ────────────────────────────────────────────────────
def _drawdown_series(cum_series):
    run_max = cum_series.cummax()
    return run_max - cum_series

ax = axes[0]
comb_cum = comb_tc   # TC-adjusted combined
dd = _drawdown_series(comb_cum)
ax.fill_between(dd.index, dd.values, color='#e74c3c', alpha=0.5)
ax.plot(dd.index, dd.values, color='#c0392b', lw=0.8)
ax.axvline(bt.loc[bt['is_val'], 'date'].min(), color='green', lw=1.5,
           linestyle=':', label='OOS start')
ax.set_title(f'Combined Drawdown (TC-adj)   Max DD = {dd.max():.5f}')
ax.set_ylabel('Drawdown')
ax.legend()
ax.grid(True, alpha=0.25)

# ── Per-ticker drawdowns ───────────────────────────────────────────────────────
ax = axes[1]
for tid, color in zip(sorted(id_to_ticker), palette_tk):
    tk    = id_to_ticker[tid]
    df_tk = bt[bt['ticker'] == tk].sort_values('date')
    cum_t = df_tk.set_index('date')['strategy_ret_tc'].cumsum()
    dd_t  = _drawdown_series(cum_t)
    ax.plot(dd_t.index, dd_t.values, color=color, lw=1.3, alpha=0.85,
            label=f'{tk} (max={dd_t.max():.5f})')
ax.axvline(bt.loc[bt['is_val'], 'date'].min(), color='green', lw=1.5, linestyle=':')
ax.set_title('Per-Ticker Drawdown (TC-adj)')
ax.set_ylabel('Drawdown')
ax.legend()
ax.grid(True, alpha=0.25)

plt.suptitle('Drawdown Analysis — MMTFv3', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_drawdown.png", dpi=150, bbox_inches='tight')
plt.show()

## 15. Monthly PnL Calendar Heatmap

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS) + 1,
                          figsize=(7 * (len(TICKERS) + 1), 5))

slices = [('ALL', bt)] + [(tk, bt[bt['ticker'] == tk])
                           for tk in sorted(id_to_ticker.values())]

for ax, (label, df_s) in zip(axes, slices):
    df_s = df_s.copy()
    df_s['year']  = df_s['date'].dt.year
    df_s['month'] = df_s['date'].dt.month

    pivot = (
        df_s.groupby(['year', 'month'])['strategy_ret_tc']
        .sum()
        .unstack('month')
    )
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'][:len(pivot.columns)]
    pivot = pivot.reindex(columns=['Jan','Feb','Mar','Apr','May','Jun',
                                   'Jul','Aug','Sep','Oct','Nov','Dec'])

    abs_max = pivot.abs().max().max()
    sns.heatmap(
        pivot, ax=ax, cmap='RdYlGn',
        center=0, vmin=-abs_max, vmax=abs_max,
        annot=True, fmt='.4f', annot_kws={'size': 7},
        linewidths=0.5, cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'{label} — Monthly PnL (TC-adj)')
    ax.set_xlabel('')

plt.suptitle('Monthly PnL Calendar Heatmap — MMTFv3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BACKTEST_PATH / f"{prefix}_bt_monthly_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## 16. Quantile Head — Class Logit Distribution

(Only rendered when `USE_PTP=True`.) Shows the distribution of raw logits for
each of the 5 return-quantile classes across IS and OOS, and the class
assignment accuracy broken down by true return decile.

In [ ]:
if USE_PTP and bt_logits is not None:
    import torch.nn.functional as F

    probs    = torch.softmax(torch.tensor(bt_logits), dim=-1).numpy()  # (N, 5)
    pred_cls = probs.argmax(axis=1)                                     # (N,)

    # True return class labels (quintile-based, consistent with training)
    from CTAFlow.models.deep_learning.multi_branch.tft.mmtf_v3_core import returns_to_classes
    true_cls = returns_to_classes(
        torch.tensor(bt_ret), n_classes=5
    ).numpy()  # (N,)

    # ── Class probability distribution ─────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    class_labels = ['Q1\n(Bearish)', 'Q2', 'Q3\n(Neutral)', 'Q4', 'Q5\n(Bullish)']
    colors_cls   = plt.cm.RdYlGn(np.linspace(0, 1, 5))
    for c in range(5):
        ax.hist(probs[:, c], bins=50, alpha=0.55, color=colors_cls[c],
                label=class_labels[c], density=True)
    ax.set_title('Class Probability Distributions (all samples)')
    ax.set_xlabel('Predicted Probability')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    # ── Confusion matrix (OOS) ───────────────────────────────────────────────
    ax = axes[1]
    oos_mask  = is_val
    conf = np.zeros((5, 5), dtype=int)
    for t, p in zip(true_cls[oos_mask], pred_cls[oos_mask]):
        conf[int(t), int(p)] += 1
    conf_norm = conf / conf.sum(axis=1, keepdims=True)
    sns.heatmap(conf_norm, ax=ax, cmap='Blues', annot=True, fmt='.2f',
                xticklabels=class_labels, yticklabels=class_labels,
                vmin=0, vmax=1)
    ax.set_title('Normalised Confusion Matrix (OOS)')
    ax.set_xlabel('Predicted Class')
    ax.set_ylabel('True Class')

    plt.suptitle('QuantileHead — Class Logit Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BACKTEST_PATH / f"{prefix}_bt_class_logits.png", dpi=150, bbox_inches='tight')
    plt.show()

    # ── Per-class accuracy report ─────────────────────────────────────────────
    print("\nOOS Classification Accuracy by Class:")
    for c in range(5):
        mask_c = true_cls[oos_mask] == c
        acc_c  = (pred_cls[oos_mask][mask_c] == c).mean() * 100 if mask_c.sum() > 0 else 0
        print(f"  {class_labels[c]}: {acc_c:.1f}%  (n={mask_c.sum()})")
    overall_acc = (pred_cls[oos_mask] == true_cls[oos_mask]).mean() * 100
    print(f"  Overall: {overall_acc:.2f}%")
else:
    print("USE_PTP=False or logits not collected — skipping class analysis.")

## 17. Export Results

In [ ]:
# ── Full backtest DataFrame ───────────────────────────────────────────────────
bt_export_path = BACKTEST_PATH / f"{prefix}_bt_full.csv"
bt.to_csv(bt_export_path, index=False)
print(f"Full backtest saved → {bt_export_path}")

# ── Summary table ─────────────────────────────────────────────────────────────
summary_path = BACKTEST_PATH / f"{prefix}_bt_summary.csv"
df_summary.to_csv(summary_path)
print(f"Summary saved       → {summary_path}")

# ── Trade-level table ─────────────────────────────────────────────────────────
trades_path = BACKTEST_PATH / f"{prefix}_bt_trades.csv"
df_trades.to_csv(trades_path)
print(f"Trade table saved   → {trades_path}")

# ── Quick final print ─────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("BACKTEST COMPLETE — KEY OOS METRICS")
print("=" * 70)
oos_all = bt[bt['is_val']]
for col_label, col in [('Gross', 'strategy_ret'), ('TC-adj', 'strategy_ret_tc')]:
    sr = oos_all[col].values
    ann_sh = (sr.mean() / (sr.std() + 1e-8)) * ANNUALIZATION
    print(f"  {col_label:8s}: Ann.Sharpe={ann_sh:.3f}   "
          f"Net PnL={sr.sum():.5f}   "
          f"Max DD={_metrics(oos_all, '', col=col)['Max Drawdown']:.5f}")